# Data Audit

Prepare the repo inside Kaggle, build grouped manifests, and inspect the starter prompt inventory before running heavier experiments.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("FRS_REPO_URL", "").strip()
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in the Kaggle environment before running this notebook.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)

In [ ]:
split_dir = Path("data/processed/splits/sample_small")
subprocess.run(
    [
        sys.executable,
        "scripts/make_splits.py",
        "--input",
        "data/processed/prompts/sample_prompts.jsonl",
        "--output-dir",
        str(split_dir),
        "--config",
        "configs/data/prompt_sets_small.yaml",
        "--seed",
        "7",
    ],
    check=True,
 )
print(split_dir)

In [ ]:
import json
from collections import Counter
import pandas as pd

records = []
with open("data/processed/prompts/sample_prompts.jsonl", "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
display(df.groupby(["group", "topic"]).size().reset_index(name="count"))
display(df.groupby("family_id").size().reset_index(name="examples_per_family"))
display(pd.DataFrame({"split": ["discovery", "selection", "holdout"], "path": [str(split_dir / "discovery.jsonl"), str(split_dir / "selection.jsonl"), str(split_dir / "holdout.jsonl")]}))